# NUTDTS 816 Time Series Analysis
## L11 Exponential smoothing: SES, Holt, Holt-Winters

Lab notebook for Chapter 6 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 6.2 Simple exponential smoothing (SES)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
import tsdata
oilp = tsdata.oil()      # annual oil production, Saudi Arabia, millions of tonnes, 1965-2013 (fpp2 'oil')
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(oilp.index, oilp.values, lw=1, marker='o', ms=3, color='#1B5E3A', label='observed')
fut = pd.date_range(oilp.index[-1], periods=6, freq='YS')[1:]
for a, col in [(0.2, '#B8860B'), (0.6, '#2F6DB5'), (0.9, '#A0302A')]:
    f = SimpleExpSmoothing(oilp, initialization_method='estimated').fit(smoothing_level=a, optimized=False)
    ax.plot(oilp.index, f.fittedvalues.values, lw=1.2, color=col, label=f'SES fitted, α = {a}')
    ax.plot(fut, [f.forecast(5).iloc[0]] * 5, color=col, lw=2)
opt = SimpleExpSmoothing(oilp, initialization_method='estimated').fit()
ax.set_title(f'SES on annual oil production with three α values (optimised α = {opt.params["smoothing_level"]:.2f}); flat forecasts to the right'); ax.legend(fontsize=8); ax.set_xlabel('')
print(pd.DataFrame({'weight on x_T-k (α = 0.2)': [0.2 * 0.8**k for k in range(6)], 'weight on x_T-k (α = 0.6)': [0.6 * 0.4**k for k in range(6)]}, index=[f'k={k}' for k in range(6)]).round(3).to_string())
_caption = 'Small α gives a smooth, slowly adapting level; large α tracks the data closely. All SES forecasts are flat at the final level.'

### 6.3 Holt's linear trend method

In [ ]:
air = tsdata.airpassengers(); ann = air.resample('YS').sum() / 1000     # annual passengers, millions, 1949-1960 (trend, no seasonality)
livestock = ann  # short annual series with a clear trend
fig, ax = plt.subplots(figsize=(9, 3.4)); ax.plot(livestock.index, livestock.values, marker='o', ms=3, lw=1, color='#1B5E3A', label='observed')
h = 8; fut = pd.date_range(livestock.index[-1], periods=h + 1, freq='YS')[1:]
for name, kw, col in [('Holt linear', {}, '#B8860B'), ('Holt damped (φ estimated)', {'damped_trend': True}, '#2F6DB5'), ('SES', None, '#555555')]:
    m = (SimpleExpSmoothing(livestock, initialization_method='estimated').fit() if kw is None else Holt(livestock, initialization_method='estimated', **kw).fit())
    ax.plot(fut, m.forecast(h).values, lw=2, color=col, label=name + (f" (φ = {m.params['damping_trend']:.2f})" if kw and kw.get('damped_trend') else ''))
ax.set_title('Annual airline passengers (millions): SES, Holt and damped Holt forecasts'); ax.legend(fontsize=8); ax.set_xlabel('')
_caption = 'SES is flat; Holt extrapolates the recent slope; the damped trend bends toward a limit. Which is right depends on how far ahead and how much you believe the trend persists.'

### 6.4 Holt-Winters seasonal methods

In [ ]:
grid = tsdata.nigeria_grid(); gtr, gte = grid[:'2025-06'], grid['2025-07':]; h = len(gte)
hw_add = ExponentialSmoothing(gtr, trend='add', seasonal='add', seasonal_periods=12, damped_trend=True, initialization_method='estimated').fit()
hw_mul = ExponentialSmoothing(gtr, trend='add', seasonal='mul', seasonal_periods=12, damped_trend=True, initialization_method='estimated').fit()
fig, ax = plt.subplots(figsize=(9, 3.4)); grid['2022':].plot(ax=ax, lw=1, label='observed')
hw_add.forecast(h).plot(ax=ax, lw=2, color='#B8860B', label='Holt-Winters additive, damped'); hw_mul.forecast(h).plot(ax=ax, lw=1.5, ls='--', color='#2F6DB5', label='Holt-Winters multiplicative, damped')
ax.axvline(gte.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('Grid generation (simulated): Holt-Winters forecasts from June 2025')
print(pd.DataFrame({'additive': hw_add.params, 'multiplicative': hw_mul.params}).loc[['smoothing_level', 'smoothing_trend', 'smoothing_seasonal', 'damping_trend']].round(3).to_string())
for name, m in [('HW additive damped', hw_add), ('HW multiplicative damped', hw_mul)]:
    f_ = m.forecast(h); print(f'{name:26s} MAE = {(f_ - gte).abs().mean():5.0f} MW  RMSE = {np.sqrt(((f_ - gte)**2).mean()):5.0f} MW')
_caption = 'On an additive series the two variants give similar forecasts. The estimated parameters show a moderately adaptive level, a fixed slope and a fixed seasonal pattern (β* and γ at zero).'

## Exercises

1. Show that SES with $\alpha = 1$ is the naive method and with $\alpha \to 0$ (and a suitable initial level) approaches the mean method. What is the ARIMA(0,1,1) $\theta$ in each case?
2. Starting from the error-correction form $\ell_t = \ell_{t-1} + \alpha e_t$ and $e_t = x_t - \ell_{t-1}$, derive $x_t - x_{t-1} = e_t - (1-\alpha)e_{t-1}$.
3. For the damped-trend method, derive the limit of $\hat x_{t+h|t}$ as $h \to \infty$.
4. Fit ETS(A,N,N), ETS(A,A,N) and ETS(A,A$_d$,N) to the `oil` series and compare AICc and the ten-year forecasts. Which would you give a government planning office, and why?

In [ ]:
# Your work here
